In [1]:
library(kableExtra)
#library(agricolae)
library(knitr)
#library(faraway)
library(XML)
library(ggplot2)
library(tidyverse)
library(readxl)
# funciones 
pregnum=function(resp,error=0.0001,peso=1,digresp=4,diger=digresp){
paste('{',peso,':NM:=',formatC(resp,digits=digresp,format = "f"),':',formatC(error,digits=diger,format = "f"),'}',sep='')
}
premch=function(resp=1,opciones,peso=1){
if(resp<1 | resp>length(opciones) ){stop("La respuesta debe ser un numero entre 1 y el numero de opciones")}  
for(i in 1:length(opciones)){
   if(i!=resp){opciones[i]=paste('~%0%',opciones[i],'',sep="")
}else{opciones[i]=paste('~=',opciones[i],'',sep="")} 
    }
gsub(" ", "",paste0( c(paste('{',peso,':MULTICHOICE:'), noquote(opciones),'}' ),collapse=""))
} 
# doc para guardar en formato xml 
NombreCat="Limpieza/parse_num"
doc=newXMLDoc()
root=newXMLNode("quiz",doc=doc)
question=newXMLNode("question",attrs = list(type="category"),parent = root)
category=newXMLNode("category",parent = question)
text=newXMLNode("text",NombreCat,parent = category)
info=newXMLNode("info",attrs = list(format="html"),parent = question)  
textInf=newXMLNode("text","texto",parent = info)
idnum=newXMLNode("idnumber",parent=question)
# funcion para agregar la pregunta al xml 
addquestion=function(NomPr,textopregunta){
questionpr=newXMLNode("question",attrs = list(type="cloze"),parent = root)
nombrepr=newXMLNode("name",parent = questionpr)
textNomPre=newXMLNode("text",NomPr,parent =nombrepr)
questiontext=newXMLNode("questiontext",attrs = list(format="html"),parent = questionpr)
textqt=newXMLNode("text",textopregunta,parent =questiontext)
generalfeedback=newXMLNode("generalfeedback",attrs = list(format="html"),parent = questionpr)
textgf=newXMLNode("text",parent = generalfeedback)
penalty=newXMLNode("penalty",0.3333333,parent=questionpr)
hidden=newXMLNode("hidden",0,parent=questionpr)
idnumber=newXMLNode("idnumber",parent=questionpr)
}

── Attaching core tidyverse packages ────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ lubridate 1.9.4     ✔ tibble    3.3.0
✔ purrr     1.0.4     ✔ tidyr     1.3.1
── Conflicts ──────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter()     masks stats::filter()
✖ dplyr::group_rows() masks kableExtra::group_rows()
✖ dplyr::lag()        masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [2]:
setwd("~/Documents/Cursos/Laboratorio")
# lectura de los datos 
library(readxl)
df=read_excel("~/Documents/Cursos/Laboratorio/DatosEstudiantes.xlsx")
head(df)

Edad,Sexo,Tcolegio,Estrato,Peso,Estatura,Afisica,satisfaccion,Nhermanos
<dbl>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<dbl>
16,Mujer,Público,1,47,1.6,Ningún día a la semana,5,2
18,Mujer,Público,1,47,158,Cinco días a la semana,7,3
16,Mujer,Público,2,59,1.71,Ningún día a la semana,8,2
16,Mujer,Público,2,59,1.71,Ningún día a la semana,8,2
17,Hombre,Privado,1,68,175,Tres días a la semana,9,4
17,Hombre,Público,1,51,1.65,Tres días a la semana,8,3


In [3]:
tabla<- df|>count(Estrato,Tcolegio)|>pivot_wider(names_from = Estrato, values_from = n, values_fill = 0)
print(tabla)

# A tibble: 2 × 4
  Tcolegio   `1`   `2`   `3`
  <chr>    <int> <int> <int>
1 Privado      2     4     3
2 Público     26    10     0


In [4]:
# cruce de Tcolegio sexo 
tab1='<style type="text/css">
.tg  {border-collapse:collapse;border-color:#aaa;border-spacing:0;}
.tg td{background-color:#fff;border-color:#aaa;border-style:solid;border-width:1px;color:#333;
  font-family:Arial, sans-serif;font-size:14px;overflow:hidden;padding:10px 5px;word-break:normal;}
.tg th{background-color:#f38630;border-color:#aaa;border-style:solid;border-width:1px;color:#fff;
  font-family:Arial, sans-serif;font-size:14px;font-weight:normal;overflow:hidden;padding:10px 5px;word-break:normal;}
.tg .tg-0pky{border-color:inherit;text-align:left;vertical-align:top}
</style>
<table class="tg"><thead>
  <tr>
    <th class="tg-0pky">Tipo Colegio</th>
    <th class="tg-0pky">Hombre </th>
    <th class="tg-0pky">Mujer</th>
  </tr></thead>
<tbody>
  <tr>
    <td class="tg-0pky">Privado</td>
    <td class="tg-0pky">res12</td>
    <td class="tg-0pky"><span style="font-weight:400;font-style:normal">res13</span></td>
  </tr>
  <tr>
    <td class="tg-0pky">Público</td>
    <td class="tg-0pky"><span style="font-weight:400;font-style:normal">res22</span></td>
    <td class="tg-0pky"><span style="font-weight:400;font-style:normal">res23</span></td>
  </tr>
</tbody>
</table>' 

res12=pregnum(as.numeric(tabla[1,2]),error=0,digresp=0)
res13=pregnum(as.numeric(tabla[1,3]),error=0,digresp=0)
res22=pregnum(as.numeric(tabla[2,2]),error=0,digresp=0)
res23=pregnum(as.numeric(tabla[2,3]),error=0,digresp=0) 
tab1=gsub(pattern = "res12", replacement = res12, x = tab1)
tab1=gsub(pattern = "res13", replacement = res13, x = tab1)
tab1=gsub(pattern = "res22", replacement = res22, x = tab1)
tab1=gsub(pattern = "res23", replacement = res23, x = tab1)

text1p=paste("El archivo <code>DatosEstudiantes.xlsx</code> contiene datos tomados de estudiantes de primer semestre durante un experimento que buscaba evaluar el nivel de satisfacción de los estudiantes con el servicio prestado por la Universidad. Haga la lectura de los datos y:")
text2p=paste("Complete la siguiente tabla que muestra la distribución de Hombres y Mujeres de acuerdo al tipo de colegio")

text3p=paste("Ayuda: si usa <code>tidyverse</code>, se pueden obtener los conteos mediante: <br> 
<pre>
<code>
tabla<- df|>count(x,y)|>pivot_wider(names_from = x, values_from = n, values_fill = 0)
print(tabla)
</code>
</pre>")


texto_preg=paste(text1p,text2p,tab1,text3p,sep='<br>')


NomPreg="Count"
semilla=1
Nombre_pre=paste(NomPreg,semilla,sep=" ") 
Nombre_pre
#texto_preg

[1] "Count 1"

In [5]:
# agrega pregunta 
addquestion(NomPr=Nombre_pre,textopregunta =texto_preg)

In [6]:
tabla<- df|>count(Estrato,Tcolegio)|>pivot_wider(names_from = Estrato, values_from = n, values_fill = 0)
print(tabla)

# A tibble: 2 × 4
  Tcolegio   `1`   `2`   `3`
  <chr>    <int> <int> <int>
1 Privado      2     4     3
2 Público     26    10     0


In [7]:
# cruce de Tcolegio estrato
tab1='<style type="text/css">
.tg  {border-collapse:collapse;border-color:#aaa;border-spacing:0;}
.tg td{background-color:#fff;border-color:#aaa;border-style:solid;border-width:1px;color:#333;
  font-family:Arial, sans-serif;font-size:14px;overflow:hidden;padding:10px 5px;word-break:normal;}
.tg th{background-color:#f38630;border-color:#aaa;border-style:solid;border-width:1px;color:#fff;
  font-family:Arial, sans-serif;font-size:14px;font-weight:normal;overflow:hidden;padding:10px 5px;word-break:normal;}
.tg .tg-baqh{text-align:center;vertical-align:top}
.tg .tg-c3ow{border-color:inherit;text-align:center;vertical-align:top}
.tg .tg-0pky{border-color:inherit;text-align:left;vertical-align:top}
.tg .tg-0lax{text-align:left;vertical-align:top}
</style>
<table class="tg"><thead>
  <tr>
    <th class="tg-0pky">Tipo Colegio</th>
    <th class="tg-c3ow">1</th>
    <th class="tg-c3ow">2</th>
    <th class="tg-baqh">3</th>
  </tr></thead>
<tbody>
  <tr>
    <td class="tg-0pky">Privado</td>
    <td class="tg-0pky">res12</td>
    <td class="tg-0pky"><span style="font-weight:400;font-style:normal">res13</span></td>
    <td class="tg-0lax">res14</td>
  </tr>
  <tr>
    <td class="tg-0pky">Público</td>
    <td class="tg-0pky"><span style="font-weight:400;font-style:normal">res22</span></td>
    <td class="tg-0pky"><span style="font-weight:400;font-style:normal">res23</span></td>
    <td class="tg-0lax">res24</td>
  </tr>
</tbody>
</table>' 

res12=pregnum(as.numeric(tabla[1,2]),error=0,digresp=0)
res13=pregnum(as.numeric(tabla[1,3]),error=0,digresp=0)
res14=pregnum(as.numeric(tabla[1,4]),error=0,digresp=0)
res22=pregnum(as.numeric(tabla[2,2]),error=0,digresp=0)
res23=pregnum(as.numeric(tabla[2,3]),error=0,digresp=0) 
res24=pregnum(as.numeric(tabla[2,4]),error=0,digresp=0) 
tab1=gsub(pattern = "res12", replacement = res12, x = tab1)
tab1=gsub(pattern = "res13", replacement = res13, x = tab1)
tab1=gsub(pattern = "res14", replacement = res14, x = tab1)
tab1=gsub(pattern = "res22", replacement = res22, x = tab1)
tab1=gsub(pattern = "res23", replacement = res23, x = tab1)
tab1=gsub(pattern = "res24", replacement = res24, x = tab1)

text1p=paste("El archivo <code>DatosEstudiantes.xlsx</code> contiene datos tomados de estudiantes de primer semestre durante un experimento que buscaba evaluar el nivel de satisfacción de los estudiantes con el servicio prestado por la Universidad. Haga la lectura de los datos y:")
text2p=paste("Complete la siguiente tabla que muestra la distribución de los estudiantes según el tipo de colegio y el estrato ")

text3p=paste("Ayuda: si usa <code>tidyverse</code>, se pueden obtener los conteos mediante: <br> 
<pre>
<code>
tabla<- df|>count(x,y)|>pivot_wider(names_from = x, values_from = n, values_fill = 0)
print(tabla)
</code>
</pre>")


texto_preg=paste(text1p,text2p,tab1,text3p,sep='<br>')


NomPreg="Count"
semilla=2
Nombre_pre=paste(NomPreg,semilla,sep=" ") 
Nombre_pre
#texto_preg

[1] "Count 2"

In [8]:
# agrega pregunta 
addquestion(NomPr=Nombre_pre,textopregunta =texto_preg)

In [9]:
saveXML(doc, file="PreguntasCount.xml")

[1] "PreguntasCount.xml"